<h1><center> <ins><b>Implied Volatility</b></ins></center></h1>

The Black-Scholes formulae for options takes as input the time to expiry, strike price, underlying asset price, interest rate all together with the volatility to output the price of the option. All these values are easily measured except volatility, meaning that in order to have an accurate model, we need to find an accurate figure for this volatility.
<br>
For this, we will think about the Black-Scholes formula differently. Suppose we can see an option with a year to expiry, strike price of 100, underlying asset price of 105 and an interest rate of 6% with an option price of 6.50. We can find a volatility such that if we use all the information above into the Black-Scholes formula, we would get a corresponding option value of 6.50. This is what is referred to as **implied volatility**. In a sense it is the market’s view of volatility over the life of the option. Assuming that we are using option prices to estimate the implied volatility then provided the option price is less than the asset and greater than zero then we can find a unique value for the implied volatility. (If the option price is outside these bounds then there’s a very extreme arbitrage opportunity.)\
So in essence, we are solving the equation 
$$V(S,X,r,q,\sigma,T,t) = x$$
Where every value is known apart from $\sigma$.$$$$
To solve this, we will use the **Newton-Raphson** method which uses derivative of the option price with respect to the volatility (the vega) in the calculation. This is an iterative method that goes as follows:
1. Define a function $f(x)$ such that we are trying to solve $f(x)=0$
2. Make an initial guess for x
3. Find next step via $x_{n+1}=x_{n}-\frac{f(x_{n})}{f'(x_{n})}$
4. Iterate until $\lvert f(x_{n})\rvert < \epsilon$ where $\epsilon$ is our degree of accuracy and the less it is, the more accurate our solution will be

We will translate this into our model and our needs:
1. Let $f(\sigma)= V_{BS_{\sigma}} - V_{M}$, where $V_{BS_{\sigma}}$ is our Option Value calculated from the Black-Scholes formula and our guess or iterated value of volatility ($\sigma$) and $V_{M}$ is the actual market Option Value
2. From [Brenner and Subrahmanyam (1988)](http://www.cfapubs.org/doi/abs/10.2469/faj.v44.n5.80), we will make our intial guess be $\sigma = \sqrt{\frac{2\pi}{T}}\frac{V_{M}}{S}$
3. Use $\sigma_{n+1}=\sigma_{n}-\frac{f(\sigma_{n})}{f'(\sigma_{n})}=\sigma_{n}-\frac{V_{BS_{\sigma_{n}}} - V_{M}}{\text{Vega}_{BS_{\sigma_{n}}}}$, where $\text{Vega}_{BS_{\sigma_{n}}}=\frac{\partial}{\partial \sigma}V_{BS_{\sigma_{n}}}$
4. Iterate until $\lvert f(x_{n})\rvert < \epsilon = 10^{-8}$

In [41]:
# Import libraries

import numpy as np
from scipy import stats

In [42]:
# Set all values apart from volatility

S = 100
K = 105
r = 0.06
q = 0
T = 1
t = 0.5
EC = 3.5
EP = 5
BC = 0.4
BP = 0.53
epsilon = 10**(-8)

In [43]:
# Define d1, d2, n(x)
def d1(S,K,r,q,sigma,T,t):
    d1=(np.log(S/K)+((r-q+((sigma**2)/2))*(T-t)))/(sigma*np.sqrt((T-t)))
    return d1
def d2(S,K,r,q,sigma,T,t):
    d2=(np.log(S/K)+(r-q-((sigma**2)/2))*(T-t))/(sigma*np.sqrt((T-t)))    
    return d2
def n(x):
    ans=np.exp(-(x**2)/2)/(np.sqrt(2*np.pi))   
    return ans

In [44]:
# Define Option Values and Vegas for the 4 types of options
def Value_EC(S,K,r,q,sigma,T,t):
    OVEC= (S*(stats.norm.cdf(d1(S,K,r,q,sigma,T,t)))*(np.exp(-q*(T-t))))-(K*(stats.norm.cdf(d2(S,K,r,q,sigma,T,t)))*(np.exp(-r*(T-t))))
    return OVEC
def Value_EP(S,K,r,q,sigma,T,t):
    OVEP= -(S*(stats.norm.cdf(-(d1(S,K,r,q,sigma,T,t))))*(np.exp(-q*(T-t))))+(K*(stats.norm.cdf(-(d2(S,K,r,q,sigma,T,t))))*(np.exp(-r*(T-t))))
    return OVEP
def Value_BC(S,K,r,q,sigma,T,t):
    OVBC= (stats.norm.cdf(d1(S,K,r,q,sigma,T,t)))*(np.exp(-r*(T-t)))
    return OVBC
def Value_BP(S,K,r,q,sigma,T,t):
    OVBP= (stats.norm.cdf(-(d1(S,K,r,q,sigma,T,t))))*(np.exp(-r*(T-t)))
    return OVBP

def Vega_EC(S,K,r,q,sigma,T,t):
    VEC = S*(np.sqrt(T-t))*(n(d1(S,K,r,q,sigma,T,t)))*(np.exp(-q*(T-t)))
    return VEC
def Vega_EP(S,K,r,q,sigma,T,t):
    VEP = S*(np.sqrt(T-t))*(n(d1(S,K,r,q,sigma,T,t)))*(np.exp(-q*(T-t)))
    return VEP
def Vega_BC(S,K,r,q,sigma,T,t):
    VBC = -((np.exp(-r*(T-t)))*(n(d2(S,K,r,q,sigma,T,t)))*((np.sqrt(T-t))+((d2(S,K,r,q,sigma,T,t))/sigma)))
    return VBC
def Vega_BP(S,K,r,q,sigma,T,t):
    VBP = ((np.exp(-r*(T-t)))*(n(d2(S,K,r,q,sigma,T,t)))*((np.sqrt(T-t))+((d2(S,K,r,q,sigma,T,t))/sigma)))
    return VBP

In [85]:
# Make a function to give us our iterated value of volatility using the Newton Raphson method for European Call Options
def ImpVol_EC(S,K,r,q,T,t,EC,epsilon,iterations=1000):
    sigma = np.sqrt((2*np.pi)/T)*(EC/S)
    for i in range(iterations):        
        difference = (Value_EC(S,K,r,q,sigma,T,t)) - EC
        if abs(difference) < epsilon:
            print(f'Found on the {i}th iteration')
            print(f'Difference is equal to {difference}')
            print()
            sigma = sigma*100
            print(f'Implied Volatility is {"%.3f"%round(sigma,3)}%')
            break        
        sigma = sigma - ((difference)/(Vega_EC(S,K,r,q,sigma,T,t)))
    return sigma

# Make a function to give us our iterated value of volatility using the Newton Raphson method for European Put Options
def ImpVol_EP(S,K,r,q,T,t,EP,epsilon,iterations=1000):
    sigma = np.sqrt((2*np.pi)/T)*(EP/S)
    for i in range(iterations):        
        difference = (Value_EP(S,K,r,q,sigma,T,t)) - EP
        if abs(difference) < epsilon:
            print(f'Found on the {i}th iteration')
            print(f'Difference is equal to {difference}')
            print()
            sigma = sigma*100
            print(f'Implied Volatility is {"%.3f"%round(sigma,3)}%')
            break        
        sigma = sigma - ((difference)/(Vega_EP(S,K,r,q,sigma,T,t)))
    return sigma

# Make a function to give us our iterated value of volatility using the Newton Raphson method for Binary Call Options
def ImpVol_BC(S,K,r,q,T,t,BC,epsilon,iterations=1000):
    sigma = np.sqrt((2*np.pi)/T)*(BC/S)
    for i in range(iterations):        
        difference = (Value_BC(S,K,r,q,sigma,T,t)) - BC
        if abs(difference) < epsilon:
            print(f'Found on the {i}th iteration')
            print(f'Difference is equal to {difference}')
            print()
            sigma = sigma*100
            print(f'Implied Volatility is {"%.3f"%round(sigma,3)}%')
            break        
        sigma = sigma - ((difference)/(Vega_BC(S,K,r,q,sigma,T,t)))
    return sigma

# Make a function to give us our iterated value of volatility using the Newton Raphson method for Binary Put Options
def ImpVol_BP(S,K,r,q,T,t,BP,epsilon,iterations=1000):
    sigma = np.sqrt((2*np.pi)/T)*(BP/S)
    for i in range(iterations):     
        difference = (Value_BP(S,K,r,q,sigma,T,t)) - BP
        if abs(difference) < epsilon:
            print(f'Found on the {i}th iteration')
            print(f'Difference is equal to {difference}')
            print()
            sigma = sigma*100
            print(f'Implied Volatility is {"%.3f"%round(sigma,3)}%')
            break    
        sigma = sigma - ((difference)/(Vega_BP(S,K,r,q,sigma,T,t)))
    return sigma

In [86]:
sigma_EC = ImpVol_EC(S,K,r,q,T,t,EC,epsilon,iterations=1000)

Found on the 3th iteration
Difference is equal to 4.263256414560601e-14

Implied Volatility is 15.400%


In [87]:
sigma_EP = ImpVol_EP(S,K,r,q,T,t,EP,epsilon,iterations=1000)

Found on the 2th iteration
Difference is equal to 3.460108644048887e-09

Implied Volatility is 13.982%


In [88]:
sigma_BC = ImpVol_BC(S,K,r,q,T,t,BC,epsilon,iterations=1000)

Found on the 16th iteration
Difference is equal to 4.082348681322401e-09

Implied Volatility is 10.287%


In [90]:
sigma_BP = ImpVol_BP(S,K,r,q,T,t,BP,epsilon,iterations=1000)

Found on the 847th iteration
Difference is equal to -9.870429362734967e-09

Implied Volatility is 15.550%
